# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Question (from weeks 1-4):** which visible pages should a reviewer open first because their CTR is far below what their position normally earns, *and will stay that way*?

**Task type:** ranking, built on a yes/no label. Label = `low_later`: the page's CTR in the **later** 30 days is under half of its position band's typical CTR in that window (a proxy I defined, not an observed "fixed it" outcome). Inputs come only from the **earlier** 30 days plus fixed page properties.

**Methods, and why each is here:**
- **My week-4 rule (the baseline).** Frozen: flag if CTR is under half the band norm and at least 10 clicks were expected; score = missing clicks. Nothing in it is tuned this week.
- **Logistic regression.** Readable first model for a yes/no label: each feature gets one weight I can inspect.
- **Gradient boosting (`HistGradientBoostingClassifier`, depth 3).** "Where safe": shallow trees, fixed seed, judged only on clients it never saw.
- **Decision tree, depth 3.** A model I can print and read. If it is close to the bigger ones, the bigger ones have not earned their complexity.
- **Permutation importance** to see what the boosting model leans on, and **error analysis** to see where it fails.

I judge every method with the same measure as week 4: **precision@K**, the share of the K highest-ranked pages that really were low later, next to the base rate.

In [1]:
import os, sys, subprocess, warnings
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| seed", SEED)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"
if IN_COLAB:
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same pages and label as the week-4 baseline
BANDS = [0, 3, 6, 10, 15, 20]
BAND_LABELS = ["1-3", "4-6", "7-10", "11-15", "16-20"]
el = df[(df["impressions_prev_30d"] >= 100) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
el["band"] = pd.cut(el["avg_position"], BANDS, labels=BAND_LABELS)
el["ctr_prev"] = 100 * el["clicks_prev_30d"] / el["impressions_prev_30d"]

ev = el[el["impressions_last_30d"] >= 100].copy().reset_index(drop=True)       # pages that have a label
ev["ctr_last"] = 100 * ev["clicks_last_30d"] / ev["impressions_last_30d"]
lt = ev.groupby("band", observed=True)[["clicks_last_30d", "impressions_last_30d"]].sum()
typ_last = 100 * lt["clicks_last_30d"] / lt["impressions_last_30d"]
ev["y"] = (ev["ctr_last"] < 0.5 * ev["band"].map(typ_last).astype(float)).astype(int)   # label: low_later

# Features knowable from the earlier window / fixed page properties only
ev["log_imp"] = np.log1p(ev["impressions_prev_30d"])
ev["log_clk"] = np.log1p(ev["clicks_prev_30d"])
ev["log_sess"] = np.log1p(ev["sessions_prev_30d"])
ev["has_wc"] = ev["word_count"].notna().astype(int)              # missing word_count follows content type -> flag, don't fill silently
ev["wc"] = ev["word_count"].fillna(0)
ev["has_kw"] = ev["search_volume"].notna().astype(int)
ev["log_sv"] = np.log1p(ev["search_volume"].fillna(0))
for v in ["informational", "transactional", "commercial"]:
    ev["int_" + v] = (ev["main_intent"] == v).astype(int)
ev["is_kw_article"] = (ev["content_type"] == "keyword article").astype(int)

STATIC = ["log_imp", "log_clk", "ctr_prev", "avg_position", "log_sess", "content_age_days", "has_wc", "wc",
          "has_kw", "log_sv", "int_informational", "int_transactional", "int_commercial", "is_kw_article"]
RELATIVE = ["ratio", "gap", "miss"]       # built per fold from TRAINING clients only (see section 2)
FEATURES = STATIC + RELATIVE

EXCLUDED = {
    "impressions_last_30d / clicks_last_30d / sessions_last_30d": "the label window",
    "trend_direction, trend_pct, is_declining_label": "label source for another question",
    "ctr, clicks_90d, impressions_90d, engagement_rate, scroll_rate, ai_*": "90-day totals include the later 30 days",
    "days_since_last_update": "an update inside the later 30 days happens AFTER the decision moment",
    "content_id, client_id": "pseudonyms: grouping only",
}
bad = [c for c in FEATURES if "last_30d" in c or c.startswith("trend_") or c in ("ctr", "days_since_last_update")]
assert not bad, f"excluded column in FEATURES: {bad}"

print(f"\nLabelled pages: {len(ev):,} from {ev['client_id'].nunique()} clients | base rate of low_later: {ev['y'].mean():.3f}")
print(f"Features ({len(FEATURES)}):", ", ".join(FEATURES))
for k, v in EXCLUDED.items():
    print(f"  excluded: {k:62} {v}")

scikit-learn 1.6.1 | pandas 2.2.3 | seed 42

Labelled pages: 11,111 from 29 clients | base rate of low_later: 0.410
Features (17): log_imp, log_clk, ctr_prev, avg_position, log_sess, content_age_days, has_wc, wc, has_kw, log_sv, int_informational, int_transactional, int_commercial, is_kw_article, ratio, gap, miss
  excluded: impressions_last_30d / clicks_last_30d / sessions_last_30d     the label window
  excluded: trend_direction, trend_pct, is_declining_label                 label source for another question
  excluded: ctr, clicks_90d, impressions_90d, engagement_rate, scroll_rate, ai_* 90-day totals include the later 30 days
  excluded: days_since_last_update                                         an update inside the later 30 days happens AFTER the decision moment
  excluded: content_id, client_id                                          pseudonyms: grouping only


## 2. Split design

**Grouped by client, five folds (`GroupKFold`).** A random split would put pages from the same client on both sides, and a model could look good just by recognising a client's habits. This way every page is scored by a model that never saw any page from its client. The picks I judge are the **out-of-fold** predictions, pooled.

Two more rules to keep it fair:
- **Benchmarks from training clients only.** The "typical CTR per position band" that feeds `ratio`, `gap`, `miss` and the baseline rule is recomputed inside each fold from the training clients and then applied to the held-out clients. The week-4 baseline used all pages for this, which is why its number there (0.82 at top 50) is a little higher than the baseline row below.
- **Not a time split.** The data has only two 30-day windows, so I cannot hold out later months. What I test is "a new client", not "a new time period".

One client holds a large share of the pages (shown below), so one fold is basically that client alone. I report how much that matters.

In [2]:
folds = list(GroupKFold(n_splits=5).split(ev, ev["y"], ev["client_id"]))
fold_of = np.zeros(len(ev), dtype=int)
rows = []
for f, (tr, te) in enumerate(folds):
    fold_of[te] = f
    rows.append({"fold": f, "train pages": len(tr), "test pages": len(te),
                 "test clients": ev.iloc[te]["client_id"].nunique(),
                 "base rate in test": round(ev.iloc[te]["y"].mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

sizes = ev["client_id"].value_counts()
BIG = sizes.index[0]
print(f"\nLargest client holds {sizes.iloc[0] / len(ev):.0%} of the labelled pages (base rate without it: "
      f"{ev.loc[ev['client_id'] != BIG, 'y'].mean():.3f})")

 fold  train pages  test pages  test clients  base rate in test
    0         6786        4325             1              0.483
    1         9414        1697             7              0.408
    2         9415        1696             6              0.272
    3         9414        1697             8              0.415
    4         9415        1696             7              0.355

Largest client holds 39% of the labelled pages (base rate without it: 0.363)


## 3. Train + compare vs my baseline

Same pages, same label, same grouped folds, same measure for every row. Seeds are fixed (`random_state=42`).

In [3]:
def band_typical(train):
    g = train.groupby("band", observed=True)[["clicks_prev_30d", "impressions_prev_30d"]].sum()
    return 100 * g["clicks_prev_30d"] / g["impressions_prev_30d"]

def add_relative(d, typical):
    d = d.copy()
    d["typ"] = d["band"].map(typical).astype(float)
    d["ratio"] = d["ctr_prev"] / d["typ"]
    d["gap"] = d["ctr_prev"] - d["typ"]
    d["expected"] = d["impressions_prev_30d"] * d["typ"] / 100
    d["miss"] = d["expected"] - d["clicks_prev_30d"]
    return d

MODELS = ["baseline", "logreg", "gboost", "tree"]
oof = {m: np.zeros(len(ev)) for m in MODELS}
gb_models, test_frames = [], []

for f, (tr, te) in enumerate(folds):
    typical = band_typical(ev.iloc[tr])                  # training clients only
    a, b = add_relative(ev.iloc[tr], typical), add_relative(ev.iloc[te], typical)

    # frozen week-4 rule
    flag = (b["ratio"] < 0.5) & (b["expected"] >= 10)
    oof["baseline"][te] = np.where(flag, b["miss"], 0.0)

    lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)).fit(a[FEATURES], a["y"])
    oof["logreg"][te] = lr.predict_proba(b[FEATURES])[:, 1]

    gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED).fit(a[FEATURES], a["y"])
    oof["gboost"][te] = gb.predict_proba(b[FEATURES])[:, 1]
    gb_models.append(gb); test_frames.append(b)

    dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED).fit(a[FEATURES], a["y"])
    oof["tree"][te] = dt.predict_proba(b[FEATURES])[:, 1]

print("Out-of-fold scores built for:", ", ".join(MODELS))

Out-of-fold scores built for: baseline, logreg, gboost, tree


In [4]:
ids = ev["content_id"].values
labels = ev["y"].values

def p_at_k(scores, k, mask=None):
    m = np.ones(len(ev), bool) if mask is None else mask
    order = np.lexsort((ids[m], -scores[m]))             # score desc, ties by id -> same order every run
    return round(float(labels[m][order[:k]].mean()), 3)

KS = (10, 20, 50, 100, 200)
not_big = (ev["client_id"] != BIG).values
rows = [{"model": "random picking (base rate)", **{f"p@{k}": round(float(labels.mean()), 3) for k in KS},
         "AUC": 0.5, "mean p@50 per fold": round(float(labels.mean()), 3), "p@50 without biggest client": round(float(labels[not_big].mean()), 3)}]
names = {"baseline": "week-4 rule (frozen)", "logreg": "logistic regression",
         "gboost": "gradient boosting (depth 3)", "tree": "decision tree (depth 3)"}
for m in MODELS:
    s = oof[m]
    per_fold = [p_at_k(s, 50, fold_of == f) for f in range(5)]
    rows.append({"model": names[m], **{f"p@{k}": p_at_k(s, k) for k in KS},
                 "AUC": round(roc_auc_score(labels, s), 3),
                 "mean p@50 per fold": round(float(np.mean(per_fold)), 3),
                 "p@50 without biggest client": p_at_k(s, 50, not_big)})
table = pd.DataFrame(rows).set_index("model")
table

,p@10,p@20,p@50,p@100,p@200,AUC,mean p@50 per fold,p@50 without biggest client
model,,,,,,,,
random picking (base rate),0.41,0.41,0.41,0.41,0.410,0.500,0.410,0.363
week-4 rule (frozen),0.90,0.80,0.76,0.75,0.755,0.551,0.680,0.740
logistic regression,0.80,0.85,0.82,0.83,0.800,0.779,0.768,0.840
gradient boosting (depth 3),1.00,1.00,1.00,0.92,0.900,0.788,0.760,0.920
decision tree (depth 3),0.80,0.80,0.70,0.71,0.695,0.771,0.716,0.700


In [5]:
# Exploratory only: a one-line rule - lowest ratio among pages with a big volume floor.
# I tried THREE floors and picked the best after seeing these labels, so treat it as optimistic.
challenger = {}
for floor in (1000, 5000, 10000):
    s = np.zeros(len(ev))
    for tr, te in folds:
        b = add_relative(ev.iloc[te], band_typical(ev.iloc[tr]))
        s[te] = np.where(b["impressions_prev_30d"] >= floor, -b["ratio"], -99.0)
    challenger[floor] = s
ch = pd.DataFrame({f"lowest ratio, >= {fl:,} impressions": {**{f"p@{k}": p_at_k(s, k) for k in (20, 50, 100)},
                                                         "p@50 without biggest client": p_at_k(s, 50, not_big)}
                   for fl, s in challenger.items()}).T
ch

,p@20,p@50,p@100,p@50 without biggest client
"lowest ratio, >= 1,000 impressions",0.50,0.58,0.69,0.64
"lowest ratio, >= 5,000 impressions",0.85,0.92,0.91,0.82
"lowest ratio, >= 10,000 impressions",1.00,0.98,0.97,0.92


**Reading the table.** Against the frozen rule, both logistic regression and gradient boosting do better at every K from 20 up, and better on average across folds. Gradient boosting looks near-perfect at the very top, but that is mostly one client: when the biggest client is left out its precision@50 falls, and per fold it is about level with the plain logistic regression. The depth-3 tree is no better than the week-4 rule at the top. And the exploratory one-line rule, which I picked after seeing results, gets close to the boosting model, so I do **not** conclude that the complex model earned its place.

**Method choice, after the results:** I would carry logistic regression forward as the model to explain (readable, steady across folds) and keep gradient boosting as a check. The week-4 rule stays frozen as the baseline; the volume floor in the last table is a lead for a future rule, not a replacement.

## 4. Errors and interpretation

**What the boosting model leans on.** Permutation importance: shuffle one feature on held-out clients and see how much the AUC drops (average over the five folds, five shuffles each). A big drop means the model needs that feature.

In [6]:
rows = []
for f, (gb, b) in enumerate(zip(gb_models, test_frames)):
    pi = permutation_importance(gb, b[FEATURES], b["y"], scoring="roc_auc", n_repeats=5, random_state=0)
    rows.append(pd.Series(pi.importances_mean, index=FEATURES))
imp = pd.concat(rows, axis=1).mean(axis=1).sort_values(ascending=False)
print("Average AUC drop when the feature is shuffled (top 8):")
print(imp.head(8).round(4).to_string())

Average AUC drop when the feature is shuffled (top 8):
ctr_prev        0.0315
miss            0.0274
log_clk         0.0162
log_imp         0.0123
avg_position    0.0099
ratio           0.0093
wc              0.0033
log_sess        0.0008


In [7]:
# Is the model's confidence honest? Observed low-later rate by band of predicted probability
ev["gb"] = oof["gboost"]
dec = ev.groupby(pd.qcut(ev["gb"], 10, duplicates="drop"), observed=True).agg(
    pages=("y", "size"), mean_predicted=("gb", "mean"), observed_rate=("y", "mean")).round(3)
print(dec.to_string())

# The depth-3 tree, fit on ALL labelled pages only so I can read it (not used for any score above)
a_all = add_relative(ev, band_typical(ev))
tree_all = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED).fit(a_all[FEATURES], a_all["y"])
print("\nDepth-3 tree, readable form:")
print(export_text(tree_all, feature_names=FEATURES))

                                 pages  mean_predicted  observed_rate
gb                                                                   
(0.00019000000000000006, 0.034]   1112           0.015          0.016
(0.034, 0.136]                    1111           0.080          0.084
(0.136, 0.247]                    1111           0.191          0.194
(0.247, 0.345]                    1111           0.296          0.314
(0.345, 0.44]                     1111           0.395          0.396
(0.44, 0.525]                     1111           0.484          0.484
(0.525, 0.596]                    1111           0.562          0.550
(0.596, 0.664]                    1111           0.629          0.623
(0.664, 0.73]                     1111           0.696          0.647
(0.73, 0.983]                     1111           0.789          0.787

Depth-3 tree, readable form:
|--- ratio <= 0.48
|   |--- ctr_prev <= 0.09
|   |   |--- wc <= 2372.50
|   |   |   |--- class: 1
|   |   |--- wc >  2372.50
|   |

In [8]:
# Where is the boosting model wrong? Top 100 picks, then three concrete wrong cases.
order = np.lexsort((ids, -ev["gb"].values))
ev["rank_gb"] = 0
ev.loc[ev.index[order], "rank_gb"] = np.arange(1, len(ev) + 1)
top100 = ev[ev["rank_gb"] <= 100]
print(f"Top 100: {int(top100['y'].sum())} correct, {int((1 - top100['y']).sum())} wrong | "
      f"largest client = {top100['client_id'].value_counts().iloc[0]} of 100 | "
      f"band 7-10 = {(top100['band'] == '7-10').sum()} of 100")

show = ["content_id", "band", "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "ctr_last", "gb",
        "days_since_last_update", "rank_gb"]
wrong_picks = top100[top100["y"] == 0].sort_values("gb", ascending=False)
print("\nWrong picks (high rank, was NOT low later):")
print(wrong_picks[show].head(3).round(3).to_string(index=False))

missed = ev[(ev["y"] == 1) & (ev["rank_gb"] > 3000)].sort_values("impressions_prev_30d", ascending=False)
print(f"\nMisses: {len(missed):,} of {int(ev['y'].sum()):,} low-later pages sit below rank 3,000. The biggest:")
print(missed[show].head(3).round(3).to_string(index=False))

Top 100: 92 correct, 8 wrong | largest client = 69 of 100 | band 7-10 = 68 of 100

Wrong picks (high rank, was NOT low later):
          content_id  band  impressions_prev_30d  clicks_prev_30d  ctr_prev  ctr_last    gb  days_since_last_update  rank_gb
content_2ad705808431  7-10                  3869                0     0.000     0.271 0.917                      22       55
content_e03c56bdbec9  7-10                 16197                6     0.037     0.147 0.910                      20       67
content_b1bc831deff1 11-15                  9998                0     0.000     0.452 0.907                      25       69

Misses: 2,458 of 4,550 low-later pages sit below rank 3,000. The biggest:
          content_id band  impressions_prev_30d  clicks_prev_30d  ctr_prev  ctr_last    gb  days_since_last_update  rank_gb
content_8c19996aa890  1-3                161284              389     0.241     0.281 0.181                      20     8432
content_cb112fce36be  4-6                124500   

**What it leans on.** The top features are the CTR itself (`ctr_prev`), the missing-clicks figure (`miss`) and the earlier click count (`log_clk`), then volume and position. That makes sense: a page with a very low CTR and a lot of impressions keeps a low CTR. It is not suspiciously perfect, and the near-duplicate CTR features (`ctr_prev`, `ratio`, `gap`, `miss`) share the credit, so each looks smaller than the group really is. Removing `avg_position` (which overlaps the later 30 days) changed the boosting model very little in my checks.

**Calibration.** The observed low-later rate rises with predicted probability, decile by decile, and tracks it closely, so the scores are usable as ranks and roughly as probabilities on these clients.

**What the errors look like.**
- **Wrong picks are mostly tiny-evidence or recently updated pages.** `content_226a...` (band 16-20) had 636 impressions and zero clicks: at that size zero clicks is weak evidence, and its CTR was 0.54% later. `content_e03c...` (16,197 impressions, 6 clicks, 0.04% CTR, rank 55) rose to 0.15% later, and it shows an update 20 days before the snapshot, so a change may have landed inside the later window. I excluded `days_since_last_update` because an update in the later 30 days happens after the decision moment, so the model cannot know it. A fix landing after the prediction looks like a wrong pick; I cannot prove that is what happened here.
- **The misses are moderate gaps on big pages that the week-4 rule caught.** `content_8c19...` sits at positions 1-3 with 161,284 impressions and a 0.24% CTR, under half the band norm of 0.56% but not extreme in absolute terms, and its CTR stayed low (0.28%) later. The boosting model ranks it 8,676th; my week-4 rule (all-page version) ranked it third. The model prefers pages that are *extremely* low; the rule's missing-clicks score also rewards large pages that are only moderately low. The two are complementary, not interchangeable.
- **Concentration.** About 70 of the model's top 100 are in positions 7-10 and the largest client supplies most of them. The result is real on these clients, but I would not promise it holds for a client unlike them.

**What I can and cannot claim.** *Observed:* on clients the model never saw, pages ranked high by logistic regression or boosting are still far below their band's norm a month later more often than the week-4 rule's picks. *Decision-support:* a ranked list of review candidates with readable reasons. *Not claimed:* that a rewrite raises CTR (this is observational data), that the complex model beats a simple volume rule (my own check says it does not clearly), or that this carries over to new time periods (I tested new clients, not new months).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.